# Experiment: Long Random Sequence Sensor Sanity Checks

## Objective

Audit the five annotated random-trajectory sessions at the raw 32-channel sensor level before drawing conclusions from trained spatial models. The notebook asks whether sampling, spatial support, repeated-position behavior, session structure, train/validation shift, and strip entry/exit behavior are internally plausible.

Success means that every calculation is reproducible from the current annotations, unsupported spatial regions remain explicit, hard integrity errors stop execution, and descriptive diagnostics never mutate data, annotations, splits, or model artifacts. Notebook 09 and its preliminary geometry are not imported.


In [1]:
# Setup and repository discovery
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_repository_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / 'analysis' / 'spatial_sequence').is_dir() and (candidate / 'experiments').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the realsense-apriltag repository.')

REPOSITORY_ROOT = find_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from analysis.spatial_sequence.sanity import run_sensor_sanity_checks

OUTPUT_DIRECTORY = REPOSITORY_ROOT / 'analysis' / 'reports' / '12_long_random_sequence_sensor_sanity_checks'
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
{'repository_root': str(REPOSITORY_ROOT), 'output_directory': str(OUTPUT_DIRECTORY)}


{'repository_root': '/nfs/turbo/coe-ahowens-nobackup/xclu/realsense-apriltag',
 'output_directory': '/nfs/turbo/coe-ahowens-nobackup/xclu/realsense-apriltag/analysis/reports/12_long_random_sequence_sensor_sanity_checks'}

## Analysis plan

1. Select each annotation's usable range and flag-2 raw kΩ readings.
2. Collapse on-paper observations to one median per occupied 0.5 cm cell.
3. Densify maps with a 2.5 cm truncated Gaussian kernel (sigma 1.25 cm), requiring at least three neighbors and ESS at least two.
4. Compare raw and within-session robust-z channel maps.
5. Fit one equal-session global PCA and five independent per-session PCAs.
6. Audit raw time traces, sampling cadence, repeated visits, and the physical-row train/validation split.
7. Compare ten-row entry and exit trajectories after grouping strip-contact encounters.

No lag scan, source-presence classifier, significance test, bandwidth sweep, or automatic session exclusion is performed.


In [2]:
# Run the complete deterministic analysis and overwrite only its named artifacts.
artifacts = run_sensor_sanity_checks(OUTPUT_DIRECTORY)
display(pd.Series(artifacts.summary, name='value').to_frame())


,value
session_count,5
channel_count,32
figure_count,39
all_timestamps_strictly_increasing,True
total_pose_missing_rows,177
kernel_support_fraction_min,0.896433
kernel_support_fraction_max,0.973251
median_repeat_visit_dispersion,0.500153
maximum_absolute_train_validation_median_shift,23.431214
maximum_normalized_train_validation_wasserstein,17.18265


## Data integrity and sampling cadence

Pose-missing smell rows remain in temporal summaries but cannot enter spatial analyses. Timestamp or sensor-integrity failures would have stopped the previous cell.


In [3]:
integrity = artifacts.tables['data_integrity']
display(integrity)


,session,session_id,usable_rows,flag2_rows,finite_sensor_rows,pose_valid_rows,pose_missing_rows,on_paper_rows,off_paper_pose_rows,duration_s,sample_interval_p01_s,sample_interval_median_s,sample_interval_p99_s,sample_interval_max_s,gaps_over_1_5_s,timestamp_strictly_increasing
0,mint-only-horizontal-run01,20260730_152754,648,648,648,641,7,632,9,360.339,0.55600,0.557,0.559,0.560,0,True
1,caret-mint-left-lavender-right-run01,20260730_153631,715,715,715,698,17,696,2,397.651,0.55500,0.557,0.559,0.559,0,True
2,caret-lavender-left-mint-right-run01,20260730_154848,705,705,705,640,65,637,3,392.082,0.55503,0.557,0.559,0.559,0,True
3,lavender-only-horizontal-run01,20260730_161358,782,782,782,727,55,721,6,434.966,0.55600,0.557,0.559,0.560,0,True
4,inverted-caret-lavender-left-mint-right-run01,20260730_163355,1088,1088,1088,1055,33,1044,11,605.388,0.55500,0.557,0.559,0.559,0,True


## Raw channel levels and time structure

The distribution figure compares absolute channel baselines and ranges. The time figure preserves every flag-2 reading, concatenates the five sessions without gaps, and draws each session as a separate colored line so no false boundary segment is introduced.


In [4]:
def figure_links(prefixes: tuple[str, ...]) -> None:
    paths = [path for path in artifacts.figures if any(path.stem.startswith(prefix) for prefix in prefixes)]
    lines = []
    for path in sorted(paths):
        relative = Path('..') / 'reports' / '12_long_random_sequence_sensor_sanity_checks' / 'figures' / path.name
        lines.append(f'- [{path.name}]({relative.as_posix()})')
    display(Markdown('\n'.join(lines)))

figure_links(('raw_channel_distributions', 'raw_readings_concatenated_time'))


- [raw_channel_distributions.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/raw_channel_distributions.png)
- [raw_readings_concatenated_time.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/raw_readings_concatenated_time.png)

## Spatial response and support

Each session has one 32-channel raw-kΩ figure and one primary robust-z figure. Raw panels use channel-specific limits fixed across sessions; robust panels share the range -3 to 3 so channel response patterns are directly comparable. White regions fail the neighbor or ESS requirement.


In [5]:
display(artifacts.tables['kernel_coverage'])
figure_links(('spatial_raw_', 'spatial_robust_', 'kernel_support_'))


,session,occupied_cells,on_paper_readings,median_readings_per_cell,supported_grid_nodes,grid_nodes,support_fraction,supported_neighbor_median,supported_ess_median
0,mint-only-horizontal-run01,497,632,1.0,2663,2916,0.913237,15.0,11.499801
1,caret-mint-left-lavender-right-run01,571,696,1.0,2701,2916,0.926269,17.0,13.413330
2,caret-lavender-left-mint-right-run01,504,637,1.0,2614,2916,0.896433,15.0,11.833181
3,lavender-only-horizontal-run01,608,721,1.0,2752,2916,0.943759,17.0,13.460906
4,inverted-caret-lavender-left-mint-right-run01,834,1044,1.0,2838,2916,0.973251,24.0,18.731119


- [kernel_support_caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/kernel_support_caret-lavender-left-mint-right-run01.png)
- [kernel_support_caret-mint-left-lavender-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/kernel_support_caret-mint-left-lavender-right-run01.png)
- [kernel_support_inverted-caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/kernel_support_inverted-caret-lavender-left-mint-right-run01.png)
- [kernel_support_lavender-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/kernel_support_lavender-only-horizontal-run01.png)
- [kernel_support_mint-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/kernel_support_mint-only-horizontal-run01.png)
- [spatial_raw_caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_raw_caret-lavender-left-mint-right-run01.png)
- [spatial_raw_caret-mint-left-lavender-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_raw_caret-mint-left-lavender-right-run01.png)
- [spatial_raw_inverted-caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_raw_inverted-caret-lavender-left-mint-right-run01.png)
- [spatial_raw_lavender-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_raw_lavender-only-horizontal-run01.png)
- [spatial_raw_mint-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_raw_mint-only-horizontal-run01.png)
- [spatial_robust_caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_robust_caret-lavender-left-mint-right-run01.png)
- [spatial_robust_caret-mint-left-lavender-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_robust_caret-mint-left-lavender-right-run01.png)
- [spatial_robust_inverted-caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_robust_inverted-caret-lavender-left-mint-right-run01.png)
- [spatial_robust_lavender-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_robust_lavender-only-horizontal-run01.png)
- [spatial_robust_mint-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/spatial_robust_mint-only-horizontal-run01.png)

## Global and session-fitted PCA

The global PCA gives each session total weight 1/5 in both standardization and covariance. Per-session fits use their own standardization and axes; their PC numbers are not interchangeable across sessions. PC signs are canonicalized by making the largest-magnitude loading positive. RGB maps use PC1/PC2/PC3 as red/green/blue after smoothing signed scores.


In [6]:
variance = artifacts.tables['pca_explained_variance']
display(variance.pivot(index='pca_fit', columns='component', values='explained_variance_ratio'))
display(artifacts.tables['pca_session_separation'])
figure_links(('pca_diagnostics_', 'pca_global_spatial_', 'pca_session_spatial_'))


component,PC1,PC2,PC3
pca_fit,,,
caret-lavender-left-mint-right-run01,0.656244,0.086486,0.053668
caret-mint-left-lavender-right-run01,0.674542,0.073346,0.054998
global,0.638166,0.244823,0.076189
inverted-caret-lavender-left-mint-right-run01,0.663524,0.085179,0.053922
lavender-only-horizontal-run01,0.694220,0.228558,0.043905
mint-only-horizontal-run01,0.655052,0.074331,0.056815


,record_type,session_a,session_b,value
0,within_session_radius,mint-only-horizontal-run01,,3.745915
1,within_session_radius,caret-mint-left-lavender-right-run01,,4.830511
2,within_session_radius,caret-lavender-left-mint-right-run01,,3.987948
3,within_session_radius,lavender-only-horizontal-run01,,6.412771
4,within_session_radius,inverted-caret-lavender-left-mint-right-run01,,3.548706
5,centroid_distance_pc1_pc3,mint-only-horizontal-run01,caret-mint-left-lavender-right-run01,4.990302
6,centroid_distance_pc1_pc3,mint-only-horizontal-run01,caret-lavender-left-mint-right-run01,6.946283
7,centroid_distance_pc1_pc3,mint-only-horizontal-run01,lavender-only-horizontal-run01,6.918577
8,centroid_distance_pc1_pc3,mint-only-horizontal-run01,inverted-caret-lavender-left-mint-right-run01,6.316427
9,centroid_distance_pc1_pc3,caret-mint-left-lavender-right-run01,caret-lavender-left-mint-right-run01,2.009776


- [pca_diagnostics_caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_diagnostics_caret-lavender-left-mint-right-run01.png)
- [pca_diagnostics_caret-mint-left-lavender-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_diagnostics_caret-mint-left-lavender-right-run01.png)
- [pca_diagnostics_global_equal-session.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_diagnostics_global_equal-session.png)
- [pca_diagnostics_inverted-caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_diagnostics_inverted-caret-lavender-left-mint-right-run01.png)
- [pca_diagnostics_lavender-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_diagnostics_lavender-only-horizontal-run01.png)
- [pca_diagnostics_mint-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_diagnostics_mint-only-horizontal-run01.png)
- [pca_global_spatial_caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_global_spatial_caret-lavender-left-mint-right-run01.png)
- [pca_global_spatial_caret-mint-left-lavender-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_global_spatial_caret-mint-left-lavender-right-run01.png)
- [pca_global_spatial_inverted-caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_global_spatial_inverted-caret-lavender-left-mint-right-run01.png)
- [pca_global_spatial_lavender-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_global_spatial_lavender-only-horizontal-run01.png)
- [pca_global_spatial_mint-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_global_spatial_mint-only-horizontal-run01.png)
- [pca_session_spatial_caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_session_spatial_caret-lavender-left-mint-right-run01.png)
- [pca_session_spatial_caret-mint-left-lavender-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_session_spatial_caret-mint-left-lavender-right-run01.png)
- [pca_session_spatial_inverted-caret-lavender-left-mint-right-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_session_spatial_inverted-caret-lavender-left-mint-right-run01.png)
- [pca_session_spatial_lavender-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_session_spatial_lavender-only-horizontal-run01.png)
- [pca_session_spatial_mint-only-horizontal-run01.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/pca_session_spatial_mint-only-horizontal-run01.png)

## Repeated-position consistency

A visit is a maximal consecutive run in one 0.5 cm cell; invalid rows, cell changes, or gaps over 1.5 seconds break visits. Only cells with at least three visits contribute. Dispersion is the scaled MAD of visit medians divided by that session/channel's cell-balanced robust scale.


In [7]:
visits = artifacts.tables['repeat_visit_dispersion']
visit_summary = visits.groupby(['session', 'channel'], sort=False)['normalized_dispersion'].median().unstack('channel')
display(visit_summary)


channel,S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,S11,S12,S13,S14,S15,S16,S17,S18,S19,S20,S21,S22,S23,S24,S25,S26,S27,S28,S29,S30,S31,S32
session,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
mint-only-horizontal-run01,0.115747,0.150331,0.860335,0.299105,0.338814,0.386740,0.852962,0.332328,0.221128,0.345218,0.455429,0.356790,0.341815,0.313897,0.190893,0.432317,0.361746,0.323672,0.178121,0.350312,0.340754,0.244802,0.779877,0.103127,0.303291,0.313730,0.615609,0.386498,0.359877,0.350101,0.600543,0.346824
caret-mint-left-lavender-right-run01,0.353237,0.349413,0.419516,0.501448,0.529652,0.775604,0.394245,0.621139,0.378105,0.736419,0.366702,0.576482,0.520941,0.571878,0.403251,0.644745,0.496445,0.567242,0.389470,0.492028,0.505588,0.495159,0.442338,0.126242,0.466718,0.509937,0.584286,0.879236,0.578060,0.639144,0.398246,0.525272
caret-lavender-left-mint-right-run01,0.488026,0.592068,0.280068,0.845131,0.460750,1.146530,0.480314,0.475139,0.617760,0.701867,0.251684,0.713136,0.495106,0.838925,0.537795,0.698561,0.392587,0.352568,0.603857,0.832005,0.854680,0.685065,0.261714,0.611540,0.344148,0.521550,0.270207,0.356275,0.640508,0.393448,0.447625,0.413952
lavender-only-horizontal-run01,0.315876,0.447322,0.097503,0.727179,0.312662,0.578394,0.061638,0.541117,0.481659,0.514706,0.040971,0.639402,0.617620,0.719239,0.060067,0.495515,0.720448,0.571505,0.092977,0.884250,0.683664,0.690082,0.093682,0.224948,0.736769,0.694708,0.085847,0.658174,0.708856,0.543786,0.070733,0.781698
inverted-caret-lavender-left-mint-right-run01,0.651279,0.596769,0.468595,0.645174,0.709663,0.436162,0.616618,0.628249,0.639115,0.441868,0.520503,0.654007,0.692037,0.707472,0.593082,0.798017,0.593266,0.602212,0.601278,0.724541,0.681602,0.533509,0.315868,0.799692,0.569916,0.506634,0.509640,0.652099,0.649359,0.545330,0.590629,0.654157


## Train/validation distribution shift

Every retained physical row is counted once. Guards are excluded. Median shifts and Wasserstein distances are normalized by the training scaled MAD; spatial support asks whether validation cells meet the fixed kernel criteria using only training cells.


In [8]:
shift = artifacts.tables['split_shift'].copy()
shift['absolute_median_shift'] = shift['robust_standardized_median_shift'].abs()
display(shift.nlargest(16, 'absolute_median_shift'))
display(artifacts.tables['split_spatial_support'])
figure_links(('train_validation_',))


,session,channel,train_rows,validation_rows,robust_standardized_median_shift,normalized_wasserstein,absolute_median_shift
118,lavender-only-horizontal-run01,S23,589,147,-23.431214,16.931148,23.431214
106,lavender-only-horizontal-run01,S11,589,147,-22.799212,15.955559,22.799212
102,lavender-only-horizontal-run01,S7,589,147,-22.322099,17.182650,22.322099
110,lavender-only-horizontal-run01,S15,589,147,-21.450806,14.348081,21.450806
126,lavender-only-horizontal-run01,S31,589,147,-21.406393,13.996480,21.406393
122,lavender-only-horizontal-run01,S27,589,147,-21.232727,14.127006,21.232727
114,lavender-only-horizontal-run01,S19,589,147,-20.439119,13.523125,20.439119
98,lavender-only-horizontal-run01,S3,589,147,-17.803851,10.907321,17.803851
37,caret-mint-left-lavender-right-run01,S6,535,134,1.486946,1.536518,1.486946
133,inverted-caret-lavender-left-mint-right-run01,S6,834,208,1.085300,1.142063,1.085300


,session,train_spatial_cells,validation_spatial_cells,validation_cells_supported_by_train,validation_support_fraction,validation_neighbor_median,validation_ess_median
0,mint-only-horizontal-run01,383,95,95,1.000000,12.0,9.649232
1,caret-mint-left-lavender-right-run01,439,125,125,1.000000,15.0,11.971116
2,caret-lavender-left-mint-right-run01,368,121,120,0.991736,13.0,10.003042
3,lavender-only-horizontal-run01,457,139,139,1.000000,16.0,12.288816
4,inverted-caret-lavender-left-mint-right-run01,673,190,190,1.000000,24.0,18.193823


- [train_validation_channel_shift.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/train_validation_channel_shift.png)
- [train_validation_spatial_support.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/train_validation_spatial_support.png)

## Strip entry/exit hysteresis

Same-source contacts separated by at most three seconds form one encounter. Entry uses the first inside row plus its preceding nine rows; exit uses the final inside row plus its following nine rows. All twenty positions must be eligible, the outer legs must be clean for the same source, and any other-source contact excludes the encounter. Readings are centered by the full-session channel median, and contributing sessions receive equal total weight.


In [9]:
encounters = artifacts.tables['hysteresis_encounters']
encounter_counts = encounters.groupby(['source', 'session', 'status']).size().unstack('status', fill_value=0)
display(encounter_counts)
display(artifacts.tables['hysteresis_distance_summary'].query('supported'))
figure_links(('hysteresis_trajectory_', 'hysteresis_distance_'))


status                                                  accepted  ineligible_row  other_source_contact  unclean_same_source
source   session                                                                                                           
lavender caret-lavender-left-mint-right-run01                  4              17                     7                    4
         caret-mint-left-lavender-right-run01                  2               3                    10                   12
         inverted-caret-lavender-left-mint-right-run01         3               8                    13                   18
         lavender-only-horizontal-run01                       11              26                     0                    6
mint     caret-lavender-left-mint-right-run01                  2              13                     4                    4
         caret-mint-left-lavender-right-run01                  3               8                    11                   14
         inverted-caret-lavender-left-mint-right-run01         8              11                    12                   12
         mint-only-horizontal-run01                           13               6                     0                   15

,source,distance_bin,channel,q25,median,q75,session_count,encounter_count,supported
0,lavender,0,S1,0.000462,0.006474,0.016960,4,20,True
1,lavender,0,S10,-0.000022,0.000019,0.000101,4,20,True
2,lavender,0,S11,-0.001347,-0.000101,0.000363,4,20,True
3,lavender,0,S12,-0.000080,0.000406,0.001111,4,20,True
4,lavender,0,S13,-0.000062,0.000375,0.001165,4,20,True
...,...,...,...,...,...,...,...,...,...
667,mint,9,S5,0.000031,0.000086,0.000822,2,4,True
668,mint,9,S6,-0.000006,-0.000004,0.000002,2,4,True
669,mint,9,S7,-0.000396,-0.000357,0.000715,2,4,True
670,mint,9,S8,-0.000061,0.000305,0.000762,2,4,True


- [hysteresis_distance_lavender.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/hysteresis_distance_lavender.png)
- [hysteresis_distance_mint.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/hysteresis_distance_mint.png)
- [hysteresis_trajectory_lavender.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/hysteresis_trajectory_lavender.png)
- [hysteresis_trajectory_mint.png](../reports/12_long_random_sequence_sensor_sanity_checks/figures/hysteresis_trajectory_mint.png)

## Interpretation guardrails

These are structural and descriptive checks, not an independent test set and not evidence that a smell-only model has learned spatial location. Large session separation, repeated-visit dispersion, split shift, or entry/exit differences are findings to investigate rather than automatic exclusion criteria. Absolute raw readings remain available in the time and distribution figures; robust transforms are used only where explicitly labeled.

## Next steps

- Review the unusually large channel/session shifts visible in the tables and time traces.
- Decide separately whether any discovered acquisition behavior warrants a new experimental protocol or model ablation.
- Keep this notebook frozen as the sensor-level audit record; do not back-propagate exploratory decisions into annotations without a separate review.
